In [ ]:
# ======================================================================================
# COMPETITIVE OFFER REPORT INGESTION + EXTRACTION PIPELINE V2
# ======================================================================================
#
# Purpose:
#   End-to-end weekly PDF ingestion and extraction pipeline for Competitive Offer Report
#   decks using:
#       - Colab Enterprise
#       - PyMuPDF / fitz
#       - BigQuery
#       - Gemini Vision via google-genai for dense visual table extraction
#
# Main improvements in this V2:
#   1. Title/header-first slide classification.
#   2. BTS title overrides:
#        - Post-Paid BTS (Watches)
#        - Post-Paid BTS (Tablet)
#        - Post-Paid BTS & Other Offers
#   3. CI headline title-first classification.
#   4. Header registry only scans title/header zone, never full body text.
#   5. TOC is QA only and never overrides classifier result.
#   6. Dense matrix pages are routed to Gemini Vision/table extraction.
#   7. Price-grid pages are routed to dedicated price grid rows.
#   8. Review queue captures low-confidence or incomplete rows.
#   9. Auto-approved review decisions are generated for clean rows.
#
# Usage:
#   1. Set CONFIG below.
#   2. Run this whole cell.
#   3. Upload one PDF when prompted, or set PDF_PATH manually.
#   4. For the first test, set USE_GEMINI_VISION = False.
#   5. Once classification QA looks good, set USE_GEMINI_VISION = True.
#
# ======================================================================================


# ======================================================================================
# SECTION 0 - INSTALLS
# ======================================================================================

import sys
import subprocess
import pkgutil

REQUIRED_PACKAGES = [
    ("fitz", "PyMuPDF"),
    ("google.cloud.bigquery", "google-cloud-bigquery"),
    ("google.genai", "google-genai"),
    ("db_dtypes", "db-dtypes"),
    ("pyarrow", "pyarrow"),
    ("dateutil", "python-dateutil"),
]

for import_name, pip_name in REQUIRED_PACKAGES:
    if pkgutil.find_loader(import_name.split(".")[0]) is None:
        print(f"Installing missing package: {pip_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

print("Package check complete.")


# ======================================================================================
# SECTION 1 - IMPORTS
# ======================================================================================

import os
import re
import io
import json
import time
import uuid
import base64
import hashlib
import textwrap
import traceback
import unicodedata
from pathlib import Path
from datetime import datetime, timezone, date
from typing import Dict, List, Any, Optional, Tuple

import fitz
import pandas as pd
import numpy as np
from dateutil import parser as date_parser

from google.cloud import bigquery
from google.cloud.exceptions import NotFound

try:
    from google import genai
    from google.genai import types as genai_types
    GOOGLE_GENAI_AVAILABLE = True
except Exception as e:
    GOOGLE_GENAI_AVAILABLE = False
    print("google-genai import failed. Gemini Vision will be disabled unless fixed.")
    print(e)

try:
    from google.colab import files as colab_files
    from google.colab import auth as colab_auth
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print("Imports complete.")


# ======================================================================================
# SECTION 2 - CONFIG
# ======================================================================================

PROJECT_ID = "prj-dbi-prd-1"
DATASET_ID = "ds_dbi_digitalmedia_automation"

# BigQuery dataset/job location.
# If your dataset is US multi-region, change this to "US".
BQ_LOCATION = "us-central1"

# Vertex / Gemini region.
VERTEX_LOCATION = "us-central1"

# Current model choice.
# You said you currently use gemini-2.5-pro.
GEMINI_MODEL = "gemini-2.5-pro"

# Set this manually if you already have a file path.
# Otherwise leave None and the notebook will prompt upload.
PDF_PATH = None

# Toggle this.
# First test should usually be False so classifier can be checked cheaply.
USE_GEMINI_VISION = False

# If True, writes all outputs to BigQuery.
WRITE_TO_BQ = True

# Duplicate action.
# Options:
#   None            -> prompt user
#   "cancel"        -> cancel if PDF hash already exists
#   "replace"       -> delete existing rows for the same pdf_hash, then reload
#   "append_anyway" -> load again with same pdf_hash but new run_id
DUPLICATE_ACTION = None

# Render settings for Gemini Vision.
RENDER_DPI_ZOOM = 2.0
PAGE_IMAGE_DIR_ROOT = "/tmp/competitive_offer_pages"

# Gemini rate limiting.
GEMINI_SLEEP_SECONDS = 0.5

# Prompt version for traceability.
PROMPT_VERSION = "competitive_offer_v2_vision_strict_json_2026_07"

# Auto approval threshold.
AUTO_APPROVE_MIN_CONFIDENCE = 0.86

# Debug flags.
PRINT_CLASSIFIER_QA = True
PRINT_SAMPLE_ROWS = True
MAX_GEMINI_PAGES_PER_RUN = 999

print("Config loaded.")
print(f"PROJECT_ID: {PROJECT_ID}")
print(f"DATASET_ID: {DATASET_ID}")
print(f"BQ_LOCATION: {BQ_LOCATION}")
print(f"VERTEX_LOCATION: {VERTEX_LOCATION}")
print(f"GEMINI_MODEL: {GEMINI_MODEL}")
print(f"USE_GEMINI_VISION: {USE_GEMINI_VISION}")
print(f"WRITE_TO_BQ: {WRITE_TO_BQ}")


# ======================================================================================
# SECTION 3 - TABLE NAMES
# ======================================================================================

TABLES = {
    # Bronze
    "bronze_pdfDecks": "sdi_competitiveOffers_bronze_pdfDecks_weekly",
    "bronze_slideRaw": "sdi_competitiveOffers_bronze_slideRaw_weekly",
    "bronze_slideLines": "sdi_competitiveOffers_bronze_slideLines_weekly",
    "bronze_detectedEntities": "sdi_competitiveOffers_bronze_detectedEntities_weekly",
    "bronze_slideTables": "sdi_competitiveOffers_bronze_slideTables_weekly",

    # Silver
    "silver_tocPageMap": "sdi_competitiveOffers_silver_tocPageMap_weekly",
    "silver_slideSections": "sdi_competitiveOffers_silver_slideSections_weekly",
    "silver_slideContentBlocks": "sdi_competitiveOffers_silver_slideContentBlocks_weekly",
    "silver_offerCandidates": "sdi_competitiveOffers_silver_offerCandidates_weekly",
    "silver_normalizedOffers": "sdi_competitiveOffers_silver_normalizedOffers_weekly",
    "silver_offerDeviceBridge": "sdi_competitiveOffers_silver_offerDeviceBridge_weekly",
    "silver_priceGridRows": "sdi_competitiveOffers_silver_priceGridRows_weekly",

    # Review
    "review_extractionReview": "sdi_competitiveOffers_review_extractionReview_weekly",
    "review_reviewDecisions": "sdi_competitiveOffers_review_reviewDecisions_weekly",
    "review_taxonomy": "sdi_competitiveOffers_review_taxonomy_weekly",
    "review_approvedExamples": "sdi_competitiveOffers_review_approvedExamples_weekly",
}

def fqtn(table_key: str) -> str:
    return f"{PROJECT_ID}.{DATASET_ID}.{TABLES[table_key]}"

print("Table map loaded.")


# ======================================================================================
# SECTION 4 - BIGQUERY CLIENT + TABLE SCHEMAS
# ======================================================================================

if IN_COLAB:
    try:
        colab_auth.authenticate_user()
        print("Colab authentication complete.")
    except Exception as e:
        print("Colab auth skipped or failed.")
        print(e)

bq_client = bigquery.Client(project=PROJECT_ID)
print(f"BigQuery client created for project: {bq_client.project}")


def sf(name, field_type, mode="NULLABLE"):
    return bigquery.SchemaField(name, field_type, mode=mode)


COMMON_FIELDS = [
    sf("deck_id", "STRING"),
    sf("run_id", "STRING"),
    sf("pdf_hash", "STRING"),
    sf("pdf_name", "STRING"),
    sf("deck_week", "DATE"),
]

SCHEMAS = {
    "bronze_pdfDecks": [
        sf("deck_id", "STRING"),
        sf("run_id", "STRING"),
        sf("pdf_hash", "STRING"),
        sf("pdf_name", "STRING"),
        sf("deck_week", "DATE"),
        sf("pdf_path", "STRING"),
        sf("page_count", "INT64"),
        sf("file_size_bytes", "INT64"),
        sf("duplicate_action", "STRING"),
        sf("pipeline_version", "STRING"),
        sf("ingestion_ts", "TIMESTAMP"),
    ],

    "bronze_slideRaw": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("page_label", "STRING"),
        sf("page_width", "FLOAT64"),
        sf("page_height", "FLOAT64"),
        sf("raw_text", "STRING"),
        sf("raw_text_len", "INT64"),
        sf("rendered_image_path", "STRING"),
        sf("extraction_ts", "TIMESTAMP"),
    ],

    "bronze_slideLines": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("block_no", "INT64"),
        sf("line_no", "INT64"),
        sf("span_count", "INT64"),
        sf("line_text", "STRING"),
        sf("line_text_norm", "STRING"),
        sf("x0", "FLOAT64"),
        sf("y0", "FLOAT64"),
        sf("x1", "FLOAT64"),
        sf("y1", "FLOAT64"),
        sf("font_size_max", "FLOAT64"),
        sf("font_size_avg", "FLOAT64"),
        sf("font_names", "STRING"),
        sf("is_bold", "BOOL"),
        sf("is_italic", "BOOL"),
        sf("is_top_zone", "BOOL"),
        sf("is_large_font", "BOOL"),
        sf("extraction_ts", "TIMESTAMP"),
    ],

    "bronze_detectedEntities": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("entity_type", "STRING"),
        sf("entity_value", "STRING"),
        sf("raw_context", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("source_method", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "bronze_slideTables": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("extractor_name", "STRING"),
        sf("extractor_model", "STRING"),
        sf("prompt_version", "STRING"),
        sf("raw_json", "STRING"),
        sf("raw_response_text", "STRING"),
        sf("extraction_status", "STRING"),
        sf("error_message", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_tocPageMap": COMMON_FIELDS + [
        sf("toc_source_page_num", "INT64"),
        sf("toc_line_text", "STRING"),
        sf("toc_label", "STRING"),
        sf("expected_page_num", "INT64"),
        sf("expected_section_id", "STRING"),
        sf("expected_section_group", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_slideSections": COMMON_FIELDS + [
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("requires_vision_extraction", "BOOL"),
        sf("classifier_method", "STRING"),
        sf("classifier_confidence", "FLOAT64"),
        sf("matched_rule", "STRING"),
        sf("matched_pattern", "STRING"),
        sf("title_header_text", "STRING"),
        sf("title_header_lines_json", "STRING"),
        sf("toc_expected_section", "STRING"),
        sf("toc_mismatch_flag", "BOOL"),
        sf("classification_debug_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_slideContentBlocks": COMMON_FIELDS + [
        sf("block_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("block_type", "STRING"),
        sf("carrier", "STRING"),
        sf("category", "STRING"),
        sf("block_title", "STRING"),
        sf("block_text", "STRING"),
        sf("source_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_offerCandidates": COMMON_FIELDS + [
        sf("candidate_id", "STRING"),
        sf("block_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("carrier", "STRING"),
        sf("brand", "STRING"),
        sf("product_name", "STRING"),
        sf("product_category", "STRING"),
        sf("offer_type", "STRING"),
        sf("offer_amount_text", "STRING"),
        sf("offer_value", "FLOAT64"),
        sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("customer_segment", "STRING"),
        sf("action_required", "STRING"),
        sf("date_start_text", "STRING"),
        sf("date_end_text", "STRING"),
        sf("condition_text", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("extraction_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("validation_errors_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_normalizedOffers": COMMON_FIELDS + [
        sf("offer_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("layout_type", "STRING"),
        sf("carrier", "STRING"),
        sf("brand", "STRING"),
        sf("product_name", "STRING"),
        sf("product_category", "STRING"),
        sf("offer_type", "STRING"),
        sf("offer_amount_text", "STRING"),
        sf("offer_value", "FLOAT64"),
        sf("currency", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("customer_segment", "STRING"),
        sf("action_required", "STRING"),
        sf("condition_text", "STRING"),
        sf("date_start", "DATE"),
        sf("date_end", "DATE"),
        sf("raw_evidence", "STRING"),
        sf("normalization_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("auto_approve_flag", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_offerDeviceBridge": COMMON_FIELDS + [
        sf("bridge_id", "STRING"),
        sf("offer_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("carrier", "STRING"),
        sf("device_name", "STRING"),
        sf("device_brand", "STRING"),
        sf("device_family", "STRING"),
        sf("device_model", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "silver_priceGridRows": COMMON_FIELDS + [
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("brand", "STRING"),
        sf("device_name", "STRING"),
        sf("device_category", "STRING"),
        sf("retail_price", "FLOAT64"),
        sf("offer_bucket", "STRING"),
        sf("customer_scenario", "STRING"),
        sf("id_requirement", "STRING"),
        sf("rate_plan_requirement", "STRING"),
        sf("price_text", "STRING"),
        sf("price_value", "FLOAT64"),
        sf("raw_evidence", "STRING"),
        sf("extraction_method", "STRING"),
        sf("confidence", "FLOAT64"),
        sf("needs_review", "BOOL"),
        sf("validation_errors_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_extractionReview": COMMON_FIELDS + [
        sf("review_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("review_reason", "STRING"),
        sf("review_status", "STRING"),
        sf("source_table", "STRING"),
        sf("raw_payload_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_reviewDecisions": COMMON_FIELDS + [
        sf("decision_id", "STRING"),
        sf("candidate_id", "STRING"),
        sf("grid_row_id", "STRING"),
        sf("page_num", "INT64"),
        sf("section_id", "STRING"),
        sf("decision_type", "STRING"),
        sf("decision_status", "STRING"),
        sf("decision_source", "STRING"),
        sf("decision_notes", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_taxonomy": COMMON_FIELDS + [
        sf("taxonomy_id", "STRING"),
        sf("taxonomy_type", "STRING"),
        sf("taxonomy_value", "STRING"),
        sf("taxonomy_display", "STRING"),
        sf("is_active", "BOOL"),
        sf("created_ts", "TIMESTAMP"),
    ],

    "review_approvedExamples": COMMON_FIELDS + [
        sf("example_id", "STRING"),
        sf("section_id", "STRING"),
        sf("example_type", "STRING"),
        sf("raw_evidence", "STRING"),
        sf("approved_payload_json", "STRING"),
        sf("created_ts", "TIMESTAMP"),
    ],
}


def ensure_dataset_and_tables():
    dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
    dataset_obj = bigquery.Dataset(dataset_ref)
    dataset_obj.location = BQ_LOCATION

    try:
        bq_client.get_dataset(dataset_ref)
        print(f"Dataset exists: {dataset_ref}")
    except NotFound:
        print(f"Creating dataset: {dataset_ref} in {BQ_LOCATION}")
        bq_client.create_dataset(dataset_obj)

    for key, schema in SCHEMAS.items():
        table_id = fqtn(key)
        try:
            bq_client.get_table(table_id)
            print(f"Table exists: {table_id}")
        except NotFound:
            print(f"Creating table: {table_id}")
            table_obj = bigquery.Table(table_id, schema=schema)
            bq_client.create_table(table_obj)

if WRITE_TO_BQ:
    ensure_dataset_and_tables()


# ======================================================================================
# SECTION 5 - GENERAL HELPERS
# ======================================================================================

NOW_TS = lambda: datetime.now(timezone.utc)

def make_hash_id(prefix: str, *parts: Any, n: int = 24) -> str:
    raw = "||".join("" if p is None else str(p) for p in parts)
    h = hashlib.sha256(raw.encode("utf-8")).hexdigest()[:n]
    return f"{prefix}_{h}"

def normalize_text(s: Any) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = s.replace("–", "-").replace("—", "-")
    s = s.replace("&", " and ")
    s = re.sub(r"[()\[\]{}:;,|]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip().lower()

def clean_line(s: Any) -> str:
    if s is None:
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r"\s+", " ", s).strip()
    return s

def safe_json(obj: Any) -> str:
    try:
        return json.dumps(obj, ensure_ascii=False, default=str)
    except Exception:
        return str(obj)

def safe_float(x):
    if x is None:
        return None
    if isinstance(x, float) and np.isnan(x):
        return None
    s = str(x).replace(",", "").replace("$", "").strip()
    if not s:
        return None
    if s.upper() in ["FREE", "ON US"]:
        return 0.0
    m = re.search(r"-?\d+(\.\d+)?", s)
    return float(m.group(0)) if m else None

def parse_date_safe(x):
    if x is None or str(x).strip() == "":
        return None
    try:
        return date_parser.parse(str(x), fuzzy=True).date()
    except Exception:
        return None

def compute_file_hash(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def infer_deck_week_from_pdf_or_filename(pdf_path: str, first_page_text: str = "") -> date:
    text = first_page_text or ""
    name = os.path.basename(pdf_path)

    # Example: May 8, 2026
    m = re.search(
        r"\b(January|February|March|April|May|June|July|August|September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        text,
        flags=re.IGNORECASE,
    )
    if m:
        return date_parser.parse(m.group(0)).date()

    # Example filename: 050826 means May 8, 2026
    m = re.search(r"(?<!\d)(\d{2})(\d{2})(\d{2})(?!\d)", name)
    if m:
        mm, dd, yy = m.groups()
        yyyy = 2000 + int(yy)
        return date(yyyy, int(mm), int(dd))

    raise ValueError("Could not infer deck_week from first page text or filename.")

def choose_pdf_path() -> str:
    global PDF_PATH

    if PDF_PATH and os.path.exists(PDF_PATH):
        print(f"Using configured PDF_PATH: {PDF_PATH}")
        return PDF_PATH

    if IN_COLAB:
        print("Please upload one Competitive Offer Report PDF.")
        uploaded = colab_files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        first_name = list(uploaded.keys())[0]
        PDF_PATH = f"/content/{first_name}"
        print(f"Uploaded PDF_PATH: {PDF_PATH}")
        return PDF_PATH

    raise ValueError("Set PDF_PATH manually when not running in Colab.")

def query_df(sql: str, params: Optional[List[bigquery.ScalarQueryParameter]] = None) -> pd.DataFrame:
    job_config = None
    if params:
        job_config = bigquery.QueryJobConfig(query_parameters=params)
    return bq_client.query(sql, job_config=job_config, location=BQ_LOCATION).to_dataframe()

def table_schema_map(table_key: str) -> Dict[str, bigquery.SchemaField]:
    table = bq_client.get_table(fqtn(table_key))
    return {field.name: field for field in table.schema}

def value_is_null(x):
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    return False

def clean_value_for_string(x):
    if value_is_null(x):
        return None
    if isinstance(x, (dict, list)):
        return safe_json(x)
    return str(x)

def coerce_df_to_bq_schema(df: pd.DataFrame, table_key: str) -> pd.DataFrame:
    if df is None or df.empty:
        return pd.DataFrame()

    schema = table_schema_map(table_key)
    out = df.copy()

    # Keep only existing target columns.
    for col in schema.keys():
        if col not in out.columns:
            out[col] = None

    out = out[list(schema.keys())]

    for col, field in schema.items():
        typ = field.field_type.upper()

        if typ == "STRING":
            out[col] = out[col].apply(clean_value_for_string)

        elif typ in ["INT64", "INTEGER"]:
            out[col] = pd.to_numeric(out[col], errors="coerce").astype("Int64")

        elif typ in ["FLOAT64", "FLOAT", "NUMERIC", "BIGNUMERIC"]:
            out[col] = pd.to_numeric(out[col], errors="coerce")

        elif typ in ["BOOL", "BOOLEAN"]:
            def to_bool(v):
                if value_is_null(v):
                    return None
                if isinstance(v, bool):
                    return v
                s = str(v).strip().lower()
                if s in ["true", "1", "yes", "y"]:
                    return True
                if s in ["false", "0", "no", "n"]:
                    return False
                return None
            out[col] = out[col].apply(to_bool)

        elif typ == "DATE":
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.date

        elif typ == "TIMESTAMP":
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)

    return out

def append_df_to_bq(df: pd.DataFrame, table_key: str):
    if df is None or df.empty:
        print(f"Skipping empty dataframe for {table_key}")
        return

    table_id = fqtn(table_key)
    clean_df = coerce_df_to_bq_schema(df, table_key)

    if clean_df.empty:
        print(f"Skipping empty cleaned dataframe for {table_key}")
        return

    job_config = bigquery.LoadJobConfig(
        schema=bq_client.get_table(table_id).schema,
        write_disposition=bigquery.WriteDisposition.WRITE_APPEND,
    )

    print(f"Loading {len(clean_df):,} rows into {table_id}")
    job = bq_client.load_table_from_dataframe(
        clean_df,
        table_id,
        job_config=job_config,
        location=BQ_LOCATION,
    )
    job.result()
    print(f"Loaded {len(clean_df):,} rows into {table_id}")

def delete_existing_pdf_rows(pdf_hash: str, deck_ids: List[str]):
    print("Deleting existing rows for replace action.")
    for table_key in TABLES.keys():
        table_id = fqtn(table_key)
        try:
            table = bq_client.get_table(table_id)
            cols = {f.name for f in table.schema}
            if "pdf_hash" not in cols:
                print(f"Skipping delete for {table_id}. No pdf_hash column.")
                continue

            sql = f"""
            DELETE FROM `{table_id}`
            WHERE pdf_hash = @pdf_hash
            """
            params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]

            bq_client.query(
                sql,
                job_config=bigquery.QueryJobConfig(query_parameters=params),
                location=BQ_LOCATION,
            ).result()

            print(f"Deleted existing rows from {table_id}")

        except Exception as e:
            print(f"Delete failed for {table_id}: {e}")


# ======================================================================================
# SECTION 6 - DUPLICATE HANDLING
# ======================================================================================

def get_existing_decks_by_hash(pdf_hash: str) -> pd.DataFrame:
    if not WRITE_TO_BQ:
        return pd.DataFrame()

    table_id = fqtn("bronze_pdfDecks")
    try:
        bq_client.get_table(table_id)
    except NotFound:
        return pd.DataFrame()

    sql = f"""
    SELECT deck_id, run_id, pdf_name, deck_week, ingestion_ts, duplicate_action
    FROM `{table_id}`
    WHERE pdf_hash = @pdf_hash
    ORDER BY ingestion_ts DESC
    """
    params = [bigquery.ScalarQueryParameter("pdf_hash", "STRING", pdf_hash)]
    return query_df(sql, params)

def resolve_duplicate_action(existing_df: pd.DataFrame) -> str:
    if existing_df.empty:
        return "new"

    print("\n" + "=" * 100)
    print("DUPLICATE PDF HASH FOUND")
    print("=" * 100)
    display(existing_df)

    action = DUPLICATE_ACTION
    if action is None:
        action = input("Choose action: cancel / replace / append_anyway: ").strip().lower()

    allowed = {"cancel", "replace", "append_anyway"}
    if action not in allowed:
        raise ValueError(f"Invalid duplicate action: {action}. Allowed: {allowed}")

    if action == "cancel":
        raise RuntimeError("Cancelled by duplicate handling.")

    print(f"Duplicate action selected: {action}")
    return action


# ======================================================================================
# SECTION 7 - PDF BRONZE EXTRACTION
# ======================================================================================

def render_page_to_png(page, out_path: str, zoom: float = 2.0):
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    pix.save(out_path)

def extract_pdf_bronze(pdf_path: str, deck_id: str, run_id: str, pdf_hash: str, deck_week: date):
    pdf_name = os.path.basename(pdf_path)
    image_dir = Path(PAGE_IMAGE_DIR_ROOT) / run_id
    image_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(pdf_path)

    slide_raw_rows = []
    slide_line_rows = []

    print("\n" + "=" * 100)
    print("BRONZE PDF EXTRACTION")
    print("=" * 100)
    print(f"PDF: {pdf_name}")
    print(f"Pages: {doc.page_count}")
    print(f"Image dir: {image_dir}")

    for page_idx in range(doc.page_count):
        page = doc[page_idx]
        page_num = page_idx + 1
        rect = page.rect
        raw_text = page.get_text("text") or ""

        image_path = str(image_dir / f"page_{page_num:03d}.png")
        render_page_to_png(page, image_path, zoom=RENDER_DPI_ZOOM)

        slide_raw_rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "page_label": str(page_num),
            "page_width": float(rect.width),
            "page_height": float(rect.height),
            "raw_text": raw_text,
            "raw_text_len": len(raw_text),
            "rendered_image_path": image_path,
            "extraction_ts": NOW_TS(),
        })

        text_dict = page.get_text("dict")
        page_height = float(rect.height)
        page_width = float(rect.width)

        for block_no, block in enumerate(text_dict.get("blocks", [])):
            if block.get("type") != 0:
                continue

            for line_no, line in enumerate(block.get("lines", [])):
                spans = line.get("spans", [])
                if not spans:
                    continue

                pieces = []
                font_sizes = []
                font_names = []
                x0s, y0s, x1s, y1s = [], [], [], []

                is_bold = False
                is_italic = False

                for span in spans:
                    txt = span.get("text", "")
                    if txt:
                        pieces.append(txt)
                    size = span.get("size")
                    if size:
                        font_sizes.append(float(size))
                    font = span.get("font", "")
                    if font:
                        font_names.append(font)
                        f_norm = font.lower()
                        if "bold" in f_norm or "black" in f_norm or "heavy" in f_norm:
                            is_bold = True
                        if "italic" in f_norm or "oblique" in f_norm:
                            is_italic = True
                    bbox = span.get("bbox", None)
                    if bbox:
                        x0s.append(float(bbox[0]))
                        y0s.append(float(bbox[1]))
                        x1s.append(float(bbox[2]))
                        y1s.append(float(bbox[3]))

                line_text = clean_line(" ".join(pieces))
                if not line_text:
                    continue

                x0 = min(x0s) if x0s else None
                y0 = min(y0s) if y0s else None
                x1 = max(x1s) if x1s else None
                y1 = max(y1s) if y1s else None

                font_size_max = max(font_sizes) if font_sizes else None
                font_size_avg = float(np.mean(font_sizes)) if font_sizes else None

                slide_line_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "page_num": page_num,
                    "block_no": int(block_no),
                    "line_no": int(line_no),
                    "span_count": len(spans),
                    "line_text": line_text,
                    "line_text_norm": normalize_text(line_text),
                    "x0": x0,
                    "y0": y0,
                    "x1": x1,
                    "y1": y1,
                    "font_size_max": font_size_max,
                    "font_size_avg": font_size_avg,
                    "font_names": ", ".join(sorted(set(font_names))),
                    "is_bold": is_bold,
                    "is_italic": is_italic,
                    "is_top_zone": bool(y0 is not None and y0 <= page_height * 0.25),
                    "is_large_font": False,
                    "extraction_ts": NOW_TS(),
                })

    doc.close()

    slide_raw_df = pd.DataFrame(slide_raw_rows)
    slide_lines_df = pd.DataFrame(slide_line_rows)

    # Mark large font within each page using quantiles.
    if not slide_lines_df.empty:
        slide_lines_df["is_large_font"] = False
        for p, g in slide_lines_df.groupby("page_num"):
            vals = pd.to_numeric(g["font_size_max"], errors="coerce").dropna()
            if len(vals) > 0:
                q80 = vals.quantile(0.80)
                idx = g[g["font_size_max"] >= q80].index
                slide_lines_df.loc[idx, "is_large_font"] = True

    print(f"slide_raw_df rows: {len(slide_raw_df):,}")
    print(f"slide_lines_df rows: {len(slide_lines_df):,}")

    return slide_raw_df, slide_lines_df


# ======================================================================================
# SECTION 8 - ENTITY DETECTION
# ======================================================================================

CARRIER_ALIASES = {
    "T-Mobile": ["T-Mobile", "T Mobile", "TMOBILE", "TMO", "Metro by T-Mobile", "Metro"],
    "Verizon": ["Verizon", "VZW", "Visible", "Total Wireless", "Straight Talk"],
    "AT&T": ["AT&T", "ATT", "Cricket"],
    "Xfinity": ["Xfinity", "Xfinity Mobile", "Comcast"],
    "Spectrum": ["Spectrum", "Spectrum Mobile"],
    "Optimum": ["Optimum", "Optimum Mobile"],
    "Boost": ["Boost", "Boost Mobile"],
    "Cricket": ["Cricket", "Cricket Wireless"],
    "Metro": ["Metro", "Metro by T-Mobile"],
    "Walmart": ["Walmart"],
    "Google Fi": ["Google Fi"],
    "Visible": ["Visible"],
    "Tello": ["Tello"],
    "Simple Mobile": ["Simple Mobile"],
    "US Mobile": ["US Mobile"],
}

DEVICE_BRANDS = [
    "iPhone", "Apple Watch", "iPad", "Galaxy", "Samsung", "Pixel", "Motorola",
    "Moto", "Razr", "TCL", "REVVL", "Franklin", "Netgear", "Inseego", "Gizmo",
]

def detect_entities(slide_raw_df: pd.DataFrame, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    rows = []
    for _, r in slide_raw_df.iterrows():
        page_num = int(r["page_num"])
        text = r.get("raw_text", "") or ""

        for carrier, aliases in CARRIER_ALIASES.items():
            for alias in aliases:
                if re.search(rf"\b{re.escape(alias)}\b", text, flags=re.IGNORECASE):
                    rows.append({
                        "deck_id": deck_id,
                        "run_id": run_id,
                        "pdf_hash": pdf_hash,
                        "pdf_name": pdf_name,
                        "deck_week": deck_week,
                        "page_num": page_num,
                        "entity_type": "carrier",
                        "entity_value": carrier,
                        "raw_context": alias,
                        "confidence": 0.90,
                        "source_method": "regex_alias",
                        "created_ts": NOW_TS(),
                    })

        for brand in DEVICE_BRANDS:
            if re.search(rf"\b{re.escape(brand)}\b", text, flags=re.IGNORECASE):
                rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "page_num": page_num,
                    "entity_type": "device_brand",
                    "entity_value": brand,
                    "raw_context": brand,
                    "confidence": 0.80,
                    "source_method": "regex_device_brand",
                    "created_ts": NOW_TS(),
                })

    df = pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame()
    print(f"detected_entities rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 9 - TOC PARSING AS QA ONLY
# ======================================================================================

def infer_expected_section_from_toc_label(label: str, expected_page_num: int, all_pages_for_label: List[int]) -> Tuple[str, str]:
    l = normalize_text(label)
    pages_sorted = sorted(all_pages_for_label)
    page_pos = pages_sorted.index(expected_page_num) if expected_page_num in pages_sorted else 0

    if "ci headlines" in l or "competitive intelligence" in l:
        if page_pos == 0:
            return "ci_headlines_postpaid", "ci_headlines"
        return "ci_headlines_prepaid", "ci_headlines"

    if "major competitive offer" in l or "major offer" in l:
        return "major_offer_changes", "major_offer_changes"

    if "apple device" in l or "apple smartphone" in l:
        return "postpaid_apple_device_offers", "postpaid"

    if "samsung" in l or "google" in l:
        return "postpaid_samsung_google_device_offers", "postpaid"

    if "2nd tier" in l or "affordable device" in l or "byod offers" in l:
        return "postpaid_motorola_affordable_byod_offers", "postpaid"

    if "consumer bts" in l:
        if page_pos == 0:
            return "postpaid_bts_watch_offers", "postpaid_bts"
        if page_pos == 1:
            return "postpaid_bts_tablet_offers", "postpaid_bts"
        return "postpaid_bts_other_offers", "postpaid_bts"

    if "service" in l and ("t mobile" in l or "verizon" in l or "at and t" in l):
        return "postpaid_service_offers", "postpaid"

    if "postpaid cable mvnos" in l or "cable mvno" in l:
        if page_pos == 0:
            return "cable_mvno_handset_offers", "postpaid_cable_mvno"
        return "cable_mvno_service_offers", "postpaid_cable_mvno"

    if "postpaid business" in l or "business" in l:
        return "business_device_offers", "business"

    if "national retail" in l:
        return "national_retail_readout", "national_retail"

    if l.strip() == "prepaid":
        return "section_divider_prepaid", "prepaid"

    if "prepaid brands current offers" in l:
        return "prepaid_brand_offers", "prepaid"

    if "prepaid flagship" in l:
        return "prepaid_flagship_offers", "prepaid"

    if "walmart" in l:
        return "walmart_prepaid_offers", "prepaid"

    return "unknown", "unknown"

def parse_toc_page_map(slide_raw_df: pd.DataFrame, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    rows = []

    toc_pages = slide_raw_df[
        slide_raw_df["raw_text"].fillna("").str.contains("Table of Contents", case=False, regex=False)
    ]

    if toc_pages.empty:
        print("No TOC page found.")
        return pd.DataFrame()

    for _, toc in toc_pages.iterrows():
        toc_page_num = int(toc["page_num"])
        lines = str(toc["raw_text"]).splitlines()

        for line in lines:
            line_clean = clean_line(line)
            if not line_clean:
                continue

            # Only parse TOC-looking lines that end with page numbers.
            tail = line_clean[-45:]
            page_nums = [int(x) for x in re.findall(r"\b\d{1,2}\b", tail)]
            page_nums = [p for p in page_nums if p >= 4]

            if not page_nums:
                continue

            # Remove dot leaders and trailing page nums for label.
            label = re.sub(r"[\.…]+", " ", line_clean)
            label = re.sub(r"(\s*\d{1,2}\s*(and|&|,)?\s*)+$", "", label, flags=re.IGNORECASE).strip()
            label = re.sub(r"^[▪❖\-\s]+", "", label).strip()

            if len(label) < 4:
                continue

            for expected_page_num in page_nums:
                expected_section_id, expected_group = infer_expected_section_from_toc_label(
                    label,
                    expected_page_num,
                    page_nums,
                )
                rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "toc_source_page_num": toc_page_num,
                    "toc_line_text": line_clean,
                    "toc_label": label,
                    "expected_page_num": expected_page_num,
                    "expected_section_id": expected_section_id,
                    "expected_section_group": expected_group,
                    "created_ts": NOW_TS(),
                })

    df = pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame()
    print(f"toc_page_map rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 10 - TITLE/HEADER-FIRST CLASSIFICATION
# ======================================================================================

UNKNOWN_SECTION = "unknown"

HEADER_SCAN_RAW_LINES = 18
HEADER_MAX_LINES = 8

def is_header_noise_line(line: str) -> bool:
    if not line:
        return True

    l = normalize_text(line)

    if len(l) <= 1:
        return True

    noise_patterns = [
        r"^\d+$",
        r"^page\s+\d+",
        r"^t mobile confidential$",
        r"^t-mobile confidential$",
        r"^market and$",
        r"^channel mciinsights$",
        r"^mciinsights$",
        r"^source:",
        r"^sources:",
        r"^note:",
        r"^notes:",
        r"^copyright",
        r"^©",
    ]

    return any(re.search(p, l, flags=re.IGNORECASE) for p in noise_patterns)

def extract_title_header_lines_for_page(page_num: int, slide_raw_df: pd.DataFrame, slide_lines_df: pd.DataFrame) -> Tuple[List[str], str, str]:
    raw_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]
    raw_text = raw_row["raw_text"] or ""
    page_height = float(raw_row["page_height"])

    page_lines = slide_lines_df[slide_lines_df["page_num"] == page_num].copy()

    candidate_lines = []

    if not page_lines.empty:
        # Primary: visual top zone and large font.
        page_lines["sort_y"] = pd.to_numeric(page_lines["y0"], errors="coerce")
        page_lines["sort_x"] = pd.to_numeric(page_lines["x0"], errors="coerce")
        page_lines = page_lines.sort_values(["sort_y", "sort_x"])

        top_zone = page_lines[
            (page_lines["y0"].fillna(99999) <= page_height * 0.28)
            | (page_lines["is_large_font"] == True)
        ]

        for _, lr in top_zone.head(20).iterrows():
            line = clean_line(lr["line_text"])
            if is_header_noise_line(line):
                continue
            if len(line) > 140:
                continue
            candidate_lines.append(line)
            if len(candidate_lines) >= HEADER_MAX_LINES:
                break

    # Fallback: first raw text lines.
    if not candidate_lines:
        for raw_line in str(raw_text).splitlines()[:HEADER_SCAN_RAW_LINES]:
            line = clean_line(raw_line)
            if is_header_noise_line(line):
                continue
            if len(line) > 140:
                continue
            candidate_lines.append(line)
            if len(candidate_lines) >= HEADER_MAX_LINES:
                break

    # De-dupe while preserving order.
    seen = set()
    deduped = []
    for x in candidate_lines:
        k = normalize_text(x)
        if k and k not in seen:
            deduped.append(x)
            seen.add(k)

    header_text = "\n".join(deduped)
    header_norm = normalize_text(" ".join(deduped))

    return deduped, header_text, header_norm


TITLE_FIRST_OVERRIDES = [
    # CI headlines.
    {
        "rule_name": "ci_headlines_postpaid_title_override",
        "section_id": "ci_headlines_postpaid",
        "patterns": [
            r"\bcompetitive intelligence headlines\b.*\bpostpaid\b",
            r"\bcompetitive intelligence headlines\b.*\bcable mvno\b",
            r"\bpostpaid\b.*\bkey highlights\b",
        ],
    },
    {
        "rule_name": "ci_headlines_prepaid_title_override",
        "section_id": "ci_headlines_prepaid",
        "patterns": [
            r"\bcompetitive intelligence headlines\b.*\bprepaid\b",
            r"\bprepaid\b.*\bmvno\b.*\bkey highlights\b",
        ],
    },

    # BTS exact titles.
    {
        "rule_name": "bts_watches_title_override",
        "section_id": "postpaid_bts_watch_offers",
        "patterns": [
            r"\bpost\s*-?\s*paid\s+bts\s+watches?\b",
            r"\bpostpaid\s+bts\s+watches?\b",
            r"\bbts\s+watches?\b",
        ],
    },
    {
        "rule_name": "bts_tablet_title_override",
        "section_id": "postpaid_bts_tablet_offers",
        "patterns": [
            r"\bpost\s*-?\s*paid\s+bts\s+tablets?\b",
            r"\bpostpaid\s+bts\s+tablets?\b",
            r"\bbts\s+tablets?\b",
        ],
    },
    {
        "rule_name": "bts_other_title_override",
        "section_id": "postpaid_bts_other_offers",
        "patterns": [
            r"\bpost\s*-?\s*paid\s+bts\s+and\s+other\s+offers\b",
            r"\bpostpaid\s+bts\s+and\s+other\s+offers\b",
            r"\bbts\s+and\s+other\s+offers\b",
        ],
    },

    # Price grid.
    {
        "rule_name": "metro_price_grid_title_override",
        "section_id": "prepaid_metro_price_grid",
        "patterns": [
            r"\bmetro price grid\b",
            r"\bmetro by t mobile\b.*\bprice grid\b",
        ],
    },
]

SLIDE_HEADER_RULES = [
    {
        "rule_name": "major_offer_changes",
        "section_id": "major_offer_changes",
        "patterns": [
            r"\bmajor offer changes\b",
            r"\bmajor competitive offer updates\b",
        ],
    },
    {
        "rule_name": "postpaid_apple_device_offers",
        "section_id": "postpaid_apple_device_offers",
        "patterns": [
            r"\bpost paid apple smartphone offers\b",
            r"\bpostpaid apple smartphone offers\b",
            r"\bapple smartphone offers\b",
            r"\bapple device offers\b",
        ],
    },
    {
        "rule_name": "postpaid_samsung_google_device_offers",
        "section_id": "postpaid_samsung_google_device_offers",
        "patterns": [
            r"\bpost paid samsung and google smartphone offers\b",
            r"\bpostpaid samsung and google smartphone offers\b",
            r"\bsamsung and google smartphone offers\b",
            r"\bsamsung.*google.*offers\b",
        ],
    },
    {
        "rule_name": "postpaid_motorola_affordable_byod_offers",
        "section_id": "postpaid_motorola_affordable_byod_offers",
        "patterns": [
            r"\bpost paid motorola affordable phones and byod offers\b",
            r"\bpostpaid motorola affordable phones and byod offers\b",
            r"\bmotorola affordable phones and byod offers\b",
        ],
    },
    {
        "rule_name": "postpaid_service_offers",
        "section_id": "postpaid_service_offers",
        "patterns": [
            r"\bpost paid service offers\b",
            r"\bpostpaid service offers\b",
            r"\bt mobile verizon at and t services\b",
            r"\bservice pricing and offers\b",
        ],
    },
    {
        "rule_name": "cable_mvno_handset_offers",
        "section_id": "cable_mvno_handset_offers",
        "patterns": [
            r"\bpostpaid cable mvno handset offers\b",
            r"\bcable mvno handset offers\b",
        ],
    },
    {
        "rule_name": "cable_mvno_service_offers",
        "section_id": "cable_mvno_service_offers",
        "patterns": [
            r"\bcable mvno service pricing and offers\b",
            r"\bcable mvno service offers\b",
        ],
    },
    {
        "rule_name": "business_device_offers",
        "section_id": "business_device_offers",
        "patterns": [
            r"\bbusiness flagship device offers\b",
            r"\bbusiness device offers\b",
        ],
    },
    {
        "rule_name": "national_retail_readout",
        "section_id": "national_retail_readout",
        "patterns": [
            r"\bnational retail\b",
            r"\bpromotions marketplace\b",
            r"\bkey highlights week ending\b",
        ],
    },
    {
        "rule_name": "prepaid_brand_offers",
        "section_id": "prepaid_brand_offers",
        "patterns": [
            r"\bprepaid brands current offers\b",
            r"\bmetro by t mobile boost mobile cricket\b",
        ],
    },
    {
        "rule_name": "prepaid_flagship_offers",
        "section_id": "prepaid_flagship_offers",
        "patterns": [
            r"\bflagship brands prepaid current offers\b",
            r"\bt mobile prepaid at and t prepaid verizon prepaid\b",
        ],
    },
    {
        "rule_name": "walmart_prepaid_offers",
        "section_id": "walmart_prepaid_offers",
        "patterns": [
            r"\bwalmart current offers\b",
            r"\bwalmart.*metro.*cricket.*boost\b",
        ],
    },
]

SECTION_LAYOUT_MAP = {
    "cover": "cover",
    "confidentiality_notice": "notice",
    "table_of_contents": "toc",
    "section_divider_ci_headlines": "divider",
    "section_divider_offer_updates": "divider",
    "section_divider_postpaid": "divider",
    "section_divider_prepaid": "divider",
    "end_slide": "end_slide",

    "ci_headlines_postpaid": "narrative_headlines",
    "ci_headlines_prepaid": "narrative_headlines",
    "major_offer_changes": "major_change_grid",

    "postpaid_apple_device_offers": "dense_matrix",
    "postpaid_samsung_google_device_offers": "dense_matrix",
    "postpaid_motorola_affordable_byod_offers": "dense_matrix",
    "postpaid_bts_watch_offers": "dense_matrix",
    "postpaid_bts_tablet_offers": "dense_matrix",
    "postpaid_bts_other_offers": "dense_matrix",
    "postpaid_service_offers": "dense_text_table",
    "cable_mvno_handset_offers": "dense_text_table",
    "cable_mvno_service_offers": "dense_text_table",
    "business_device_offers": "dense_matrix",
    "business_newline_byod_offers": "dense_matrix",
    "national_retail_readout": "narrative_readout",
    "prepaid_brand_offers": "dense_prepaid_table",
    "prepaid_metro_price_grid": "price_grid",
    "prepaid_flagship_offers": "dense_prepaid_table",
    "walmart_prepaid_offers": "dense_prepaid_table",
    "unknown": "unknown",
}

VISION_LAYOUT_TYPES = {
    "dense_matrix",
    "price_grid",
    "dense_prepaid_table",
    "dense_text_table",
}

def match_rule_group(rules: List[Dict[str, Any]], header_norm: str) -> Optional[Dict[str, str]]:
    for rule in rules:
        for pattern in rule["patterns"]:
            if re.search(pattern, header_norm, flags=re.IGNORECASE):
                return {
                    "section_id": rule["section_id"],
                    "rule_name": rule["rule_name"],
                    "matched_pattern": pattern,
                }
    return None

def priority_classify(page_num: int, raw_text: str, header_norm: str) -> Optional[Dict[str, str]]:
    full_norm = normalize_text(raw_text)
    raw_lines = [clean_line(x) for x in str(raw_text).splitlines() if clean_line(x)]

    if page_num == 1 and re.search(r"\bcompetitive offer report\b", full_norm):
        return {
            "section_id": "cover",
            "rule_name": "priority_cover",
            "matched_pattern": "page 1 competitive offer report",
        }

    if re.search(r"\bnotice of confidentiality\b", full_norm) or re.search(r"\bdesignated t mobile confidential\b", full_norm):
        return {
            "section_id": "confidentiality_notice",
            "rule_name": "priority_confidentiality_notice",
            "matched_pattern": "confidentiality notice",
        }

    if re.search(r"\btable of contents\b", full_norm) and re.search(r"\bci headlines\b|\bcompetitive offer updates\b", full_norm):
        return {
            "section_id": "table_of_contents",
            "rule_name": "priority_table_of_contents",
            "matched_pattern": "table of contents",
        }

    # Divider pages.
    if re.search(r"^ci headlines$", header_norm) or (
        "ci headlines" in header_norm and len(raw_lines) <= 8
    ):
        return {
            "section_id": "section_divider_ci_headlines",
            "rule_name": "priority_ci_headlines_divider",
            "matched_pattern": "sparse CI Headlines divider",
        }

    if "competitive offer updates" in header_norm and len(raw_lines) <= 10:
        return {
            "section_id": "section_divider_offer_updates",
            "rule_name": "priority_offer_updates_divider",
            "matched_pattern": "sparse Competitive Offer Updates divider",
        }

    if "competitive offers and promotional updates postpaid" in header_norm:
        return {
            "section_id": "section_divider_postpaid",
            "rule_name": "priority_postpaid_divider",
            "matched_pattern": "Competitive Offers and Promotional Updates Postpaid",
        }

    if "competitive offers and promotional updates prepaid" in header_norm:
        return {
            "section_id": "section_divider_prepaid",
            "rule_name": "priority_prepaid_divider",
            "matched_pattern": "Competitive Offers and Promotional Updates Prepaid",
        }

    if len(raw_lines) <= 5 and re.search(r"\bare you with us\b|\bthank you\b|\bquestions\b", full_norm):
        return {
            "section_id": "end_slide",
            "rule_name": "priority_end_slide",
            "matched_pattern": "sparse end slide",
        }

    return None

def classify_business_subsection(section_id: str, raw_text: str, header_norm: str) -> str:
    if section_id != "business_device_offers":
        return section_id

    text_norm = normalize_text(raw_text)

    has_device_rows = bool(re.search(r"\biphone\b|\bgalaxy\b|\bsamsung\b|\bpixel\b", text_norm))
    has_newline_byod = bool(re.search(r"\bnew line offers\b|\bbyod offers\b", text_norm))

    # May page 21 is clearly New Line/BYOD only.
    if has_newline_byod and not has_device_rows:
        return "business_newline_byod_offers"

    # April page 20 may contain both, but device rows dominate the slide.
    return "business_device_offers"

def classify_slide(
    page_num: int,
    slide_raw_df: pd.DataFrame,
    slide_lines_df: pd.DataFrame,
    toc_expected_by_page: Dict[int, str],
) -> Dict[str, Any]:

    raw_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]
    raw_text = raw_row["raw_text"] or ""

    title_lines, title_header_text, header_norm = extract_title_header_lines_for_page(
        page_num,
        slide_raw_df,
        slide_lines_df,
    )

    toc_expected = toc_expected_by_page.get(page_num)

    debug = {
        "page_num": page_num,
        "title_lines": title_lines,
        "header_norm": header_norm,
        "toc_expected_section": toc_expected,
        "steps": [],
    }

    # 1. Priority rules.
    priority = priority_classify(page_num, raw_text, header_norm)
    if priority:
        section_id = priority["section_id"]
        debug["steps"].append({"step": "priority", "matched": True, **priority})
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "priority_rule",
            "classifier_confidence": 0.99,
            "matched_rule": priority["rule_name"],
            "matched_pattern": priority["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected,
            "toc_mismatch_flag": bool(toc_expected and toc_expected != section_id),
            "classification_debug_json": safe_json(debug),
        }

    debug["steps"].append({"step": "priority", "matched": False})

    # 2. Title-first overrides.
    title_match = match_rule_group(TITLE_FIRST_OVERRIDES, header_norm)
    if title_match:
        section_id = title_match["section_id"]
        section_id = classify_business_subsection(section_id, raw_text, header_norm)
        debug["steps"].append({"step": "title_first_override", "matched": True, **title_match})
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "title_first_override",
            "classifier_confidence": 0.98,
            "matched_rule": title_match["rule_name"],
            "matched_pattern": title_match["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected,
            "toc_mismatch_flag": bool(toc_expected and toc_expected != section_id),
            "classification_debug_json": safe_json(debug),
        }

    debug["steps"].append({"step": "title_first_override", "matched": False})

    # 3. Header registry on title/header zone only.
    header_match = match_rule_group(SLIDE_HEADER_RULES, header_norm)
    if header_match:
        section_id = header_match["section_id"]
        section_id = classify_business_subsection(section_id, raw_text, header_norm)
        debug["steps"].append({"step": "header_registry_title_zone_only", "matched": True, **header_match})
        layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
        return {
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
            "requires_vision_extraction": layout_type in VISION_LAYOUT_TYPES,
            "classifier_method": "header_registry_title_zone_only",
            "classifier_confidence": 0.94,
            "matched_rule": header_match["rule_name"],
            "matched_pattern": header_match["matched_pattern"],
            "title_header_text": title_header_text,
            "title_header_lines_json": safe_json(title_lines),
            "toc_expected_section": toc_expected,
            "toc_mismatch_flag": bool(toc_expected and toc_expected != section_id),
            "classification_debug_json": safe_json(debug),
        }

    debug["steps"].append({"step": "header_registry_title_zone_only", "matched": False})

    # 4. Unknown.
    section_id = UNKNOWN_SECTION
    layout_type = SECTION_LAYOUT_MAP.get(section_id, "unknown")
    debug["steps"].append({"step": "unknown", "matched": False})

    return {
        "page_num": page_num,
        "section_id": section_id,
        "layout_type": layout_type,
        "requires_vision_extraction": False,
        "classifier_method": "unknown",
        "classifier_confidence": 0.0,
        "matched_rule": None,
        "matched_pattern": None,
        "title_header_text": title_header_text,
        "title_header_lines_json": safe_json(title_lines),
        "toc_expected_section": toc_expected,
        "toc_mismatch_flag": bool(toc_expected and toc_expected != section_id),
        "classification_debug_json": safe_json(debug),
    }

def classify_all_slides(slide_raw_df, slide_lines_df, toc_map_df, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    toc_expected_by_page = {}
    if toc_map_df is not None and not toc_map_df.empty:
        for _, r in toc_map_df.iterrows():
            toc_expected_by_page[int(r["expected_page_num"])] = r["expected_section_id"]

    rows = []
    for page_num in sorted(slide_raw_df["page_num"].unique()):
        result = classify_slide(
            page_num=page_num,
            slide_raw_df=slide_raw_df,
            slide_lines_df=slide_lines_df,
            toc_expected_by_page=toc_expected_by_page,
        )
        result.update({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "created_ts": NOW_TS(),
        })
        rows.append(result)

    df = pd.DataFrame(rows)

    print(f"slide_sections rows: {len(df):,}")

    if PRINT_CLASSIFIER_QA:
        print("\n" + "=" * 100)
        print("CLASSIFIER QA")
        print("=" * 100)
        qa_cols = [
            "page_num", "section_id", "layout_type", "classifier_method",
            "classifier_confidence", "toc_expected_section", "toc_mismatch_flag",
            "title_header_text",
        ]
        display(df[qa_cols].sort_values("page_num"))

        print("\nClassifier counts:")
        display(df.groupby(["section_id", "layout_type"]).size().reset_index(name="rows").sort_values("page_num" if "page_num" in df.columns else "rows"))

    return df


# ======================================================================================
# SECTION 11 - DETERMINISTIC CONTENT BLOCKS
# ======================================================================================

NO_CHANGE_PATTERNS = [
    r"\bno changes\b",
    r"\bno updates\b",
    r"\bno new offers\b",
    r"\bno offers expired\b",
    r"\bno expired offers\b",
    r"\bno offers ended\b",
    r"\bno offer\b",
    r"\bnone\b",
]

def is_no_change_text(text: str) -> bool:
    t = normalize_text(text)
    return any(re.search(p, t, flags=re.IGNORECASE) for p in NO_CHANGE_PATTERNS)

def infer_carrier_from_line(line: str) -> Optional[str]:
    l = normalize_text(line)
    carrier_patterns = [
        ("T-Mobile", r"\bt mobile\b|\bt-mobile\b"),
        ("AT&T", r"\bat and t\b|\batt\b"),
        ("Verizon", r"\bverizon\b|\bvzw\b"),
        ("Xfinity", r"\bxfinity\b|\bcomcast\b"),
        ("Spectrum", r"\bspectrum\b"),
        ("Metro", r"\bmetro\b"),
        ("Boost", r"\bboost\b"),
        ("Cricket", r"\bcricket\b"),
        ("Total Wireless", r"\btotal wireless\b"),
        ("Straight Talk", r"\bstraight talk\b"),
        ("Visible", r"\bvisible\b"),
        ("Google Fi", r"\bgoogle fi\b"),
        ("Tello", r"\btello\b"),
        ("Simple Mobile", r"\bsimple mobile\b"),
        ("Walmart", r"\bwalmart\b"),
    ]
    for carrier, pat in carrier_patterns:
        if re.search(pat, l):
            return carrier
    return None

def make_content_block(deck_id, run_id, pdf_hash, pdf_name, deck_week, page_num, section_id, layout_type,
                       block_type, carrier, category, block_title, block_text, source_method, confidence, needs_review):
    block_id = make_hash_id("block", deck_id, page_num, section_id, block_type, carrier, category, block_title, block_text)
    return {
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "pdf_name": pdf_name,
        "deck_week": deck_week,
        "block_id": block_id,
        "page_num": page_num,
        "section_id": section_id,
        "layout_type": layout_type,
        "block_type": block_type,
        "carrier": carrier,
        "category": category,
        "block_title": block_title,
        "block_text": block_text,
        "source_method": source_method,
        "confidence": confidence,
        "needs_review": needs_review,
        "created_ts": NOW_TS(),
    }

def build_ci_headline_blocks(raw_text: str, base: Dict[str, Any]) -> List[Dict[str, Any]]:
    lines = [clean_line(x) for x in raw_text.splitlines() if clean_line(x)]
    blocks = []
    current_carrier = None
    current_lines = []

    def flush():
        nonlocal current_carrier, current_lines, blocks
        if current_carrier and current_lines:
            txt = "\n".join(current_lines)
            blocks.append(make_content_block(
                **base,
                block_type="ci_headline",
                carrier=current_carrier,
                category="headline",
                block_title=current_carrier,
                block_text=txt,
                source_method="deterministic_ci_numbered_grouping",
                confidence=0.90,
                needs_review=False,
            ))

    for line in lines:
        m = re.match(r"^\s*(\d{1,2})\.\s+(.+?)\s*$", line)
        if m:
            flush()
            current_carrier = clean_line(m.group(2))
            current_lines = []
            continue

        if current_carrier:
            current_lines.append(line)

    flush()
    return blocks

def build_simple_bullet_blocks(raw_text: str, base: Dict[str, Any], block_type: str) -> List[Dict[str, Any]]:
    lines = [clean_line(x) for x in raw_text.splitlines() if clean_line(x)]
    blocks = []
    current_carrier = None
    current_category = None

    for line in lines:
        possible_carrier = infer_carrier_from_line(line)
        if possible_carrier and len(line) <= 40:
            current_carrier = possible_carrier
            continue

        if re.match(r"^[A-Z][A-Z\s&/()]+:$", line):
            current_category = line.strip(":").title()
            continue

        if re.match(r"^[•▪o\-]\s*", line) or re.search(r"\$\d+|\bfree\b|\bon us\b|\bno changes\b|\bexpired\b|\bnew\b", line, flags=re.IGNORECASE):
            txt = re.sub(r"^[•▪o\-]\s*", "", line).strip()
            if len(txt) < 3:
                continue
            blocks.append(make_content_block(
                **base,
                block_type=block_type,
                carrier=current_carrier,
                category=current_category,
                block_title=current_category or current_carrier or block_type,
                block_text=txt,
                source_method=f"deterministic_{block_type}_bullet_grouping",
                confidence=0.72 if current_carrier else 0.55,
                needs_review=not bool(current_carrier),
            ))

    return blocks

def build_content_blocks(slide_raw_df, slide_sections_df, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    rows = []

    for _, sec in slide_sections_df.iterrows():
        page_num = int(sec["page_num"])
        section_id = sec["section_id"]
        layout_type = sec["layout_type"]
        raw_text = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]["raw_text"] or ""

        base = {
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "section_id": section_id,
            "layout_type": layout_type,
        }

        if section_id in ["cover", "confidentiality_notice", "table_of_contents", "section_divider_ci_headlines",
                          "section_divider_offer_updates", "section_divider_postpaid", "section_divider_prepaid", "end_slide"]:
            rows.append(make_content_block(
                **base,
                block_type="non_offer_slide",
                carrier=None,
                category=None,
                block_title=section_id,
                block_text=raw_text[:1500],
                source_method="non_offer_slide_capture",
                confidence=0.99,
                needs_review=False,
            ))
            continue

        if section_id in ["ci_headlines_postpaid", "ci_headlines_prepaid"]:
            rows.extend(build_ci_headline_blocks(raw_text, base))
            continue

        if section_id == "major_offer_changes":
            rows.extend(build_simple_bullet_blocks(raw_text, base, block_type="major_offer_change"))
            continue

        if layout_type in VISION_LAYOUT_TYPES:
            # Do not create many noisy generic blocks. Create one page-level block and let Vision extract.
            rows.append(make_content_block(
                **base,
                block_type="vision_page_candidate",
                carrier=None,
                category=None,
                block_title=section_id,
                block_text=raw_text[:3000],
                source_method="vision_routing_placeholder",
                confidence=0.80,
                needs_review=not USE_GEMINI_VISION,
            ))
            continue

        if section_id == "national_retail_readout":
            rows.extend(build_simple_bullet_blocks(raw_text, base, block_type="national_retail_readout"))
            continue

        if section_id == "unknown":
            rows.append(make_content_block(
                **base,
                block_type="unknown_page",
                carrier=None,
                category=None,
                block_title="unknown",
                block_text=raw_text[:3000],
                source_method="unknown_page_capture",
                confidence=0.0,
                needs_review=True,
            ))
            continue

        rows.extend(build_simple_bullet_blocks(raw_text, base, block_type="generic_offer_text"))

    df = pd.DataFrame(rows)
    print(f"content_blocks rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 12 - GEMINI VISION EXTRACTION
# ======================================================================================

def make_gemini_client():
    if not GOOGLE_GENAI_AVAILABLE:
        raise RuntimeError("google-genai is not available.")

    # Vertex AI mode.
    client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=VERTEX_LOCATION,
    )
    return client

def page_image_bytes(path: str) -> bytes:
    with open(path, "rb") as f:
        return f.read()

def build_vision_prompt(section_id: str, layout_type: str, title_header_text: str, raw_text: str) -> str:
    allowed_carriers = [
        "T-Mobile", "Verizon", "AT&T", "Xfinity", "Spectrum", "Optimum",
        "Metro", "Boost", "Cricket", "Total Wireless", "Straight Talk",
        "Visible", "Google Fi", "Tello", "Simple Mobile", "Walmart",
    ]

    prompt = f"""
You are extracting structured competitive telecom offer data from one rendered slide image.

You must use the slide image as the primary source of truth.
Use raw text only as a secondary aid because PDF text can flatten columns incorrectly.

Return valid JSON only. No markdown. No explanation.

Slide metadata:
section_id: {section_id}
layout_type: {layout_type}
title_header_text:
{title_header_text}

Allowed carriers:
{json.dumps(allowed_carriers)}

Rules:
1. Do not invent offers.
2. Every extracted row must have raw_evidence copied or closely paraphrased from the slide.
3. Preserve carrier ownership based on visual table columns.
4. If a cell says "No offer", "No changes", "none", or "--", return offer_type "no_offer".
5. If the slide is a price grid, populate price_grid_rows and leave rows empty.
6. If the slide is not a price grid, populate rows and leave price_grid_rows empty.
7. Use null when a value is not visible.
8. Use confidence from 0 to 1.
9. Set needs_review true when carrier, product/device, offer amount, or condition is unclear.

Return this exact JSON shape:
{{
  "extraction_summary": {{
    "section_id": "{section_id}",
    "layout_type": "{layout_type}",
    "confidence": 0.0,
    "notes": null
  }},
  "rows": [
    {{
      "carrier": null,
      "brand": null,
      "product_name": null,
      "product_category": null,
      "category": null,
      "offer_type": null,
      "offer_amount_text": null,
      "offer_value": null,
      "currency": "USD",
      "rate_plan_requirement": null,
      "customer_segment": null,
      "action_required": null,
      "date_start_text": null,
      "date_end_text": null,
      "condition_text": null,
      "raw_evidence": null,
      "confidence": 0.0,
      "needs_review": true
    }}
  ],
  "price_grid_rows": [
    {{
      "brand": null,
      "device_name": null,
      "device_category": null,
      "retail_price": null,
      "offer_bucket": null,
      "customer_scenario": null,
      "id_requirement": null,
      "rate_plan_requirement": null,
      "price_text": null,
      "price_value": null,
      "raw_evidence": null,
      "confidence": 0.0,
      "needs_review": true
    }}
  ]
}}

Raw text from PDF extraction:
{raw_text[:12000]}
"""
    return textwrap.dedent(prompt).strip()

def parse_json_response(text: str) -> Dict[str, Any]:
    if not text:
        raise ValueError("Empty Gemini response.")

    t = text.strip()
    t = re.sub(r"^```json\s*", "", t, flags=re.IGNORECASE)
    t = re.sub(r"^```\s*", "", t)
    t = re.sub(r"\s*```$", "", t)

    # Try exact JSON first.
    try:
        return json.loads(t)
    except Exception:
        pass

    # Try to extract object boundaries.
    start = t.find("{")
    end = t.rfind("}")
    if start >= 0 and end > start:
        return json.loads(t[start:end + 1])

    raise ValueError("Could not parse Gemini response as JSON.")

def call_gemini_vision_for_page(page_row: pd.Series, sec_row: pd.Series) -> Dict[str, Any]:
    client = make_gemini_client()

    image_path = page_row["rendered_image_path"]
    img_bytes = page_image_bytes(image_path)

    section_id = sec_row["section_id"]
    layout_type = sec_row["layout_type"]
    title_header_text = sec_row["title_header_text"] or ""
    raw_text = page_row["raw_text"] or ""

    prompt = build_vision_prompt(
        section_id=section_id,
        layout_type=layout_type,
        title_header_text=title_header_text,
        raw_text=raw_text,
    )

    image_part = genai_types.Part.from_bytes(
        data=img_bytes,
        mime_type="image/png",
    )

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=[prompt, image_part],
        config=genai_types.GenerateContentConfig(
            temperature=0.0,
            response_mime_type="application/json",
        ),
    )

    response_text = getattr(response, "text", None)
    if not response_text and hasattr(response, "output_text"):
        response_text = response.output_text

    parsed = parse_json_response(response_text)

    return {
        "parsed_json": parsed,
        "raw_response_text": response_text,
    }

def run_vision_extraction(slide_raw_df, slide_sections_df, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    table_rows = []
    offer_candidate_rows = []
    price_grid_rows = []

    if not USE_GEMINI_VISION:
        print("USE_GEMINI_VISION is False. Skipping Gemini Vision extraction.")
        return pd.DataFrame(table_rows), pd.DataFrame(offer_candidate_rows), pd.DataFrame(price_grid_rows)

    vision_pages = slide_sections_df[
        slide_sections_df["layout_type"].isin(VISION_LAYOUT_TYPES)
        & (~slide_sections_df["section_id"].isin(["unknown"]))
    ].copy()

    if vision_pages.empty:
        print("No pages require Gemini Vision extraction.")
        return pd.DataFrame(table_rows), pd.DataFrame(offer_candidate_rows), pd.DataFrame(price_grid_rows)

    print("\n" + "=" * 100)
    print("GEMINI VISION EXTRACTION")
    print("=" * 100)
    print(f"Pages queued: {len(vision_pages):,}")

    processed = 0

    for _, sec in vision_pages.sort_values("page_num").iterrows():
        if processed >= MAX_GEMINI_PAGES_PER_RUN:
            print(f"Reached MAX_GEMINI_PAGES_PER_RUN={MAX_GEMINI_PAGES_PER_RUN}. Stopping.")
            break

        page_num = int(sec["page_num"])
        page_row = slide_raw_df[slide_raw_df["page_num"] == page_num].iloc[0]

        print("\n" + "-" * 100)
        print(f"Gemini page {page_num}: {sec['section_id']} | {sec['layout_type']}")
        print("-" * 100)

        status = "success"
        err = None
        parsed_json = None
        raw_response_text = None

        try:
            result = call_gemini_vision_for_page(page_row, sec)
            parsed_json = result["parsed_json"]
            raw_response_text = result["raw_response_text"]

            rows = parsed_json.get("rows", []) or []
            grid_rows = parsed_json.get("price_grid_rows", []) or []

            print(f"Gemini rows: {len(rows):,}")
            print(f"Gemini price_grid_rows: {len(grid_rows):,}")

            for i, row in enumerate(rows):
                candidate_id = make_hash_id(
                    "cand",
                    deck_id, page_num, sec["section_id"],
                    row.get("carrier"), row.get("product_name"),
                    row.get("offer_type"), row.get("raw_evidence"), i,
                )
                block_id = make_hash_id("block", deck_id, page_num, sec["section_id"], "gemini_vision", i)

                validation_errors = validate_offer_candidate_dict(row, sec["section_id"])

                confidence = safe_float(row.get("confidence")) if row.get("confidence") is not None else 0.75
                needs_review = bool(row.get("needs_review", False)) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

                offer_candidate_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "candidate_id": candidate_id,
                    "block_id": block_id,
                    "page_num": page_num,
                    "section_id": sec["section_id"],
                    "layout_type": sec["layout_type"],
                    "carrier": row.get("carrier"),
                    "brand": row.get("brand"),
                    "product_name": row.get("product_name"),
                    "product_category": row.get("product_category"),
                    "offer_type": row.get("offer_type"),
                    "offer_amount_text": row.get("offer_amount_text"),
                    "offer_value": safe_float(row.get("offer_value")) if row.get("offer_value") is not None else extract_offer_value(row.get("offer_amount_text")),
                    "currency": row.get("currency") or "USD",
                    "rate_plan_requirement": row.get("rate_plan_requirement"),
                    "customer_segment": row.get("customer_segment"),
                    "action_required": row.get("action_required"),
                    "date_start_text": row.get("date_start_text"),
                    "date_end_text": row.get("date_end_text"),
                    "condition_text": row.get("condition_text"),
                    "raw_evidence": row.get("raw_evidence"),
                    "extraction_method": "gemini_vision",
                    "confidence": confidence,
                    "needs_review": needs_review,
                    "validation_errors_json": safe_json(validation_errors),
                    "created_ts": NOW_TS(),
                })

            for i, gr in enumerate(grid_rows):
                grid_row_id = make_hash_id(
                    "grid",
                    deck_id, page_num, sec["section_id"],
                    gr.get("brand"), gr.get("device_name"),
                    gr.get("offer_bucket"), gr.get("price_text"), i,
                )

                validation_errors = validate_price_grid_row_dict(gr)
                confidence = safe_float(gr.get("confidence")) if gr.get("confidence") is not None else 0.75
                needs_review = bool(gr.get("needs_review", False)) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

                price_grid_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "grid_row_id": grid_row_id,
                    "page_num": page_num,
                    "section_id": sec["section_id"],
                    "brand": gr.get("brand"),
                    "device_name": gr.get("device_name"),
                    "device_category": gr.get("device_category"),
                    "retail_price": safe_float(gr.get("retail_price")),
                    "offer_bucket": gr.get("offer_bucket"),
                    "customer_scenario": gr.get("customer_scenario"),
                    "id_requirement": gr.get("id_requirement"),
                    "rate_plan_requirement": gr.get("rate_plan_requirement"),
                    "price_text": gr.get("price_text"),
                    "price_value": safe_float(gr.get("price_value")) if gr.get("price_value") is not None else safe_float(gr.get("price_text")),
                    "raw_evidence": gr.get("raw_evidence"),
                    "extraction_method": "gemini_vision_price_grid",
                    "confidence": confidence,
                    "needs_review": needs_review,
                    "validation_errors_json": safe_json(validation_errors),
                    "created_ts": NOW_TS(),
                })

        except Exception as e:
            status = "error"
            err = traceback.format_exc()
            print(f"Gemini extraction failed for page {page_num}: {e}")
            print(err)

        table_rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "page_num": page_num,
            "section_id": sec["section_id"],
            "layout_type": sec["layout_type"],
            "extractor_name": "gemini_vision_page_extractor",
            "extractor_model": GEMINI_MODEL,
            "prompt_version": PROMPT_VERSION,
            "raw_json": safe_json(parsed_json),
            "raw_response_text": raw_response_text,
            "extraction_status": status,
            "error_message": err,
            "created_ts": NOW_TS(),
        })

        processed += 1
        time.sleep(GEMINI_SLEEP_SECONDS)

    return pd.DataFrame(table_rows), pd.DataFrame(offer_candidate_rows), pd.DataFrame(price_grid_rows)


# ======================================================================================
# SECTION 13 - OFFER CANDIDATES FROM DETERMINISTIC BLOCKS
# ======================================================================================

def extract_offer_value(text: str) -> Optional[float]:
    if not text:
        return None
    t = str(text)

    if re.search(r"\bfree\b|\bon us\b|\$0\b", t, flags=re.IGNORECASE):
        return 0.0

    # Prefer explicit "$X off".
    m = re.search(r"\$([\d,]+(?:\.\d+)?)\s*(?:off|credit|bill credit|visa|mastercard|gift card|gc)?", t, flags=re.IGNORECASE)
    if m:
        return float(m.group(1).replace(",", ""))

    # "<$800", ">$830", "up to $800".
    m = re.search(r"[<>]?\s*\$([\d,]+(?:\.\d+)?)", t)
    if m:
        return float(m.group(1).replace(",", ""))

    return None

def infer_offer_type(text: str) -> str:
    t = normalize_text(text)

    if is_no_change_text(t):
        return "no_offer"
    if re.search(r"\bbogo\b|\bbuy 1\b|\bget 2nd\b", t):
        return "bogo"
    if re.search(r"\btrade\b|\btrade in\b|\btrade-in\b|\btrd\b", t):
        return "trade_in_discount"
    if re.search(r"\bport\b|\bport in\b|\bport-in\b", t):
        return "port_in_discount"
    if re.search(r"\bbyod\b|\bbring your own\b", t):
        return "byod_credit"
    if re.search(r"\bbill credit\b|\bcredits\b|\bcredit\b", t):
        return "bill_credit"
    if re.search(r"\bfree\b|\bon us\b|\$0\b", t):
        return "free_or_on_us"
    if re.search(r"\$\d+", t):
        return "discount"
    if re.search(r"\bexpired\b|\bremoved\b|\bended\b", t):
        return "expired_or_removed"
    return "other"

def extract_product_name(text: str) -> Optional[str]:
    if not text:
        return None

    device_patterns = [
        r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+Air|\s+e)?",
        r"Galaxy\s+S\d+\s*(?:Ultra|FE|Edge|\+)?",
        r"Galaxy\s+Z\s*(?:Flip|Fold)\d*",
        r"Galaxy\s+A\d+\s*5G?",
        r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?",
        r"moto\s+g\s+stylus\s+5G\s+\d{4}",
        r"moto\s+g\s+power\s+5G\s+\d{4}",
        r"moto\s+g\s+5G\s+\d{4}",
        r"moto\s+razr\s+\d{4}",
        r"Apple\s+Watch\s+[A-Za-z0-9\s]+",
        r"iPad\s+\(?A16\)?",
        r"Galaxy\s+Tab\s+[A-Za-z0-9+\s]+",
        r"TCL\s+[A-Za-z0-9\s]+",
        r"REVVL\s+[A-Za-z0-9\s]+",
    ]

    for pat in device_patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            return clean_line(m.group(0))

    # Fallback: take text before colon if short.
    first = clean_line(str(text).split(":")[0])
    if 2 <= len(first) <= 60 and not re.search(r"\boffer\b|\bdiscount\b|\btrade\b", normalize_text(first)):
        return first

    return None

def validate_offer_candidate_dict(row: Dict[str, Any], section_id: str) -> List[str]:
    errors = []

    carrier = row.get("carrier")
    raw_evidence = row.get("raw_evidence")
    offer_type = row.get("offer_type")

    if not carrier:
        errors.append("missing_carrier")
    if not raw_evidence:
        errors.append("missing_raw_evidence")
    if not offer_type:
        errors.append("missing_offer_type")

    # For no_offer rows, product may be legitimately missing.
    if offer_type != "no_offer":
        if not row.get("product_name") and not row.get("category") and not row.get("offer_amount_text"):
            errors.append("missing_product_or_offer_amount")

    return errors

def validate_price_grid_row_dict(row: Dict[str, Any]) -> List[str]:
    errors = []
    if not row.get("device_name"):
        errors.append("missing_device_name")
    if not row.get("price_text") and row.get("price_value") is None:
        errors.append("missing_price")
    if not row.get("offer_bucket"):
        errors.append("missing_offer_bucket")
    if not row.get("raw_evidence"):
        errors.append("missing_raw_evidence")
    return errors

def build_offer_candidates_from_blocks(content_blocks_df: pd.DataFrame, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    rows = []

    if content_blocks_df.empty:
        return pd.DataFrame()

    for _, b in content_blocks_df.iterrows():
        block_type = b["block_type"]
        section_id = b["section_id"]
        layout_type = b["layout_type"]
        block_text = b["block_text"] or ""

        # Skip non-offer slides and vision placeholders.
        if block_type in ["non_offer_slide", "vision_page_candidate", "unknown_page"]:
            continue

        # CI headlines are useful content but not normalized offers.
        if block_type == "ci_headline":
            continue

        offer_type = infer_offer_type(block_text)
        product_name = extract_product_name(block_text)
        offer_value = extract_offer_value(block_text)

        row = {
            "carrier": b.get("carrier"),
            "brand": None,
            "product_name": product_name,
            "product_category": None,
            "offer_type": offer_type,
            "offer_amount_text": block_text if offer_value is not None or offer_type in ["free_or_on_us", "bill_credit", "discount", "trade_in_discount", "port_in_discount"] else None,
            "offer_value": offer_value,
            "currency": "USD",
            "rate_plan_requirement": None,
            "customer_segment": None,
            "action_required": None,
            "date_start_text": None,
            "date_end_text": None,
            "condition_text": block_text,
            "raw_evidence": block_text,
            "confidence": b.get("confidence", 0.65),
            "needs_review": b.get("needs_review", True),
        }

        validation_errors = validate_offer_candidate_dict(row, section_id)
        confidence = safe_float(row["confidence"]) or 0.65
        needs_review = bool(row["needs_review"]) or bool(validation_errors) or confidence < AUTO_APPROVE_MIN_CONFIDENCE

        candidate_id = make_hash_id(
            "cand",
            deck_id, b["page_num"], section_id,
            row.get("carrier"), product_name, offer_type, block_text,
        )

        rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "candidate_id": candidate_id,
            "block_id": b["block_id"],
            "page_num": int(b["page_num"]),
            "section_id": section_id,
            "layout_type": layout_type,
            "carrier": row["carrier"],
            "brand": row["brand"],
            "product_name": row["product_name"],
            "product_category": row["product_category"],
            "offer_type": row["offer_type"],
            "offer_amount_text": row["offer_amount_text"],
            "offer_value": row["offer_value"],
            "currency": row["currency"],
            "rate_plan_requirement": row["rate_plan_requirement"],
            "customer_segment": row["customer_segment"],
            "action_required": row["action_required"],
            "date_start_text": row["date_start_text"],
            "date_end_text": row["date_end_text"],
            "condition_text": row["condition_text"],
            "raw_evidence": row["raw_evidence"],
            "extraction_method": "deterministic_block_parser",
            "confidence": confidence,
            "needs_review": needs_review,
            "validation_errors_json": safe_json(validation_errors),
            "created_ts": NOW_TS(),
        })

    df = pd.DataFrame(rows)
    print(f"deterministic offer_candidates rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 14 - NORMALIZATION + DEVICE BRIDGE
# ======================================================================================

def infer_device_brand(device_name: str) -> Optional[str]:
    if not device_name:
        return None
    t = normalize_text(device_name)
    if "iphone" in t or "ipad" in t or "apple watch" in t:
        return "Apple"
    if "galaxy" in t or "samsung" in t:
        return "Samsung"
    if "pixel" in t:
        return "Google"
    if "moto" in t or "motorola" in t or "razr" in t:
        return "Motorola"
    if "tcl" in t:
        return "TCL"
    if "revvl" in t:
        return "T-Mobile"
    if "gizmo" in t:
        return "Verizon"
    if "franklin" in t:
        return "Franklin"
    if "netgear" in t:
        return "Netgear"
    if "inseego" in t:
        return "Inseego"
    return None

def infer_product_category(product_name: str, section_id: str, raw_evidence: str) -> Optional[str]:
    t = normalize_text(" ".join([product_name or "", section_id or "", raw_evidence or ""]))

    if "watch" in t:
        return "watch"
    if "tablet" in t or "ipad" in t or "tab " in t:
        return "tablet"
    if "byod" in t:
        return "byod"
    if "hint" in t or "gateway" in t or "internet" in t:
        return "home_internet"
    if "hotspot" in t or "mifi" in t or "router" in t:
        return "hotspot_router"
    if "service" in t or "rate plan" in t or "plan " in t:
        return "service_plan"
    if "phone" in t or "iphone" in t or "galaxy" in t or "pixel" in t or "moto" in t:
        return "smartphone"

    return None

def normalize_offers(offer_candidates_df: pd.DataFrame, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    if offer_candidates_df is None or offer_candidates_df.empty:
        print("No offer candidates to normalize.")
        return pd.DataFrame()

    rows = []
    for _, c in offer_candidates_df.iterrows():
        offer_id = make_hash_id("offer", c["candidate_id"], c.get("carrier"), c.get("product_name"), c.get("offer_type"), c.get("raw_evidence"))

        product_name = c.get("product_name")
        product_category = c.get("product_category") or infer_product_category(product_name, c.get("section_id"), c.get("raw_evidence"))

        confidence = safe_float(c.get("confidence")) or 0.0
        needs_review = bool(c.get("needs_review"))
        auto_approve_flag = bool((not needs_review) and confidence >= AUTO_APPROVE_MIN_CONFIDENCE and c.get("offer_type") != "no_offer")

        rows.append({
            "deck_id": deck_id,
            "run_id": run_id,
            "pdf_hash": pdf_hash,
            "pdf_name": pdf_name,
            "deck_week": deck_week,
            "offer_id": offer_id,
            "candidate_id": c["candidate_id"],
            "page_num": int(c["page_num"]),
            "section_id": c["section_id"],
            "layout_type": c["layout_type"],
            "carrier": c.get("carrier"),
            "brand": c.get("brand") or infer_device_brand(product_name),
            "product_name": product_name,
            "product_category": product_category,
            "offer_type": c.get("offer_type"),
            "offer_amount_text": c.get("offer_amount_text"),
            "offer_value": safe_float(c.get("offer_value")),
            "currency": c.get("currency") or "USD",
            "rate_plan_requirement": c.get("rate_plan_requirement"),
            "customer_segment": c.get("customer_segment"),
            "action_required": c.get("action_required"),
            "condition_text": c.get("condition_text"),
            "date_start": parse_date_safe(c.get("date_start_text")),
            "date_end": parse_date_safe(c.get("date_end_text")),
            "raw_evidence": c.get("raw_evidence"),
            "normalization_method": "rule_normalizer_v2",
            "confidence": confidence,
            "needs_review": needs_review,
            "auto_approve_flag": auto_approve_flag,
            "created_ts": NOW_TS(),
        })

    df = pd.DataFrame(rows)
    print(f"normalized_offers rows: {len(df):,}")
    print(f"auto_approve_flag rows: {df['auto_approve_flag'].sum() if not df.empty else 0:,}")
    return df

def extract_device_names_from_text(text: str) -> List[str]:
    if not text:
        return []
    found = []

    patterns = [
        r"iPhone\s+\d+[A-Za-z]*(?:\s+Pro\s+Max|\s+Pro|\s+Plus|\s+Air|\s+e)?",
        r"Galaxy\s+S\d+\s*(?:Ultra|FE|Edge|\+)?",
        r"Galaxy\s+Z\s*(?:Flip|Fold)\d*",
        r"Galaxy\s+A\d+\s*5G?",
        r"Galaxy\s+Tab\s+[A-Za-z0-9+\s]+",
        r"Pixel\s+\d+[A-Za-z]*(?:\s+Pro\s+XL|\s+Pro|\s+Fold)?",
        r"moto\s+g\s+stylus\s+5G\s+\d{4}",
        r"moto\s+g\s+power\s+5G\s+\d{4}",
        r"moto\s+g\s+5G\s+\d{4}",
        r"moto\s+razr\s+\d{4}",
        r"Razr\+?",
        r"Apple\s+Watch\s+[A-Za-z0-9\s]+",
        r"iPad\s+\(?A16\)?",
        r"TCL\s+[A-Za-z0-9\s]+",
        r"REVVL\s+[A-Za-z0-9\s]+",
        r"SYNCUP\s+[A-Za-z0-9\s]+",
        r"GizmoWatch\s+\d*",
        r"Inseego\s+MiFi\s+[A-Za-z0-9\s]+",
        r"Netgear\s+Nighthawk\s+[A-Za-z0-9\s]+",
        r"Franklin\s+[A-Za-z0-9\s]+",
    ]

    for pat in patterns:
        for m in re.finditer(pat, text, flags=re.IGNORECASE):
            val = clean_line(m.group(0))
            if val and len(val) <= 80:
                found.append(val)

    # Dedupe.
    out = []
    seen = set()
    for x in found:
        k = normalize_text(x)
        if k not in seen:
            out.append(x)
            seen.add(k)
    return out

def build_device_bridge(normalized_offers_df: pd.DataFrame, deck_id, run_id, pdf_hash, pdf_name, deck_week):
    rows = []

    if normalized_offers_df is None or normalized_offers_df.empty:
        print("No normalized offers for device bridge.")
        return pd.DataFrame()

    for _, o in normalized_offers_df.iterrows():
        text = " ".join([
            str(o.get("product_name") or ""),
            str(o.get("raw_evidence") or ""),
        ])

        device_names = []
        if o.get("product_name"):
            device_names.append(o.get("product_name"))

        device_names.extend(extract_device_names_from_text(text))

        # Dedupe.
        clean_devices = []
        seen = set()
        for d in device_names:
            k = normalize_text(d)
            if k and k not in seen:
                clean_devices.append(d)
                seen.add(k)

        for d in clean_devices:
            bridge_id = make_hash_id("bridge", o["offer_id"], d)

            rows.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "bridge_id": bridge_id,
                "offer_id": o["offer_id"],
                "candidate_id": o["candidate_id"],
                "page_num": int(o["page_num"]),
                "section_id": o["section_id"],
                "carrier": o.get("carrier"),
                "device_name": d,
                "device_brand": infer_device_brand(d),
                "device_family": infer_product_category(d, o.get("section_id"), o.get("raw_evidence")),
                "device_model": d,
                "raw_evidence": o.get("raw_evidence"),
                "confidence": o.get("confidence"),
                "created_ts": NOW_TS(),
            })

    df = pd.DataFrame(rows).drop_duplicates(subset=["bridge_id"]) if rows else pd.DataFrame()
    print(f"offer_device_bridge rows: {len(df):,}")
    return df


# ======================================================================================
# SECTION 15 - REVIEW QUEUE + AUTO APPROVAL DECISIONS
# ======================================================================================

def build_review_tables(offer_candidates_df: pd.DataFrame, price_grid_rows_df: pd.DataFrame, normalized_offers_df: pd.DataFrame,
                        deck_id, run_id, pdf_hash, pdf_name, deck_week):
    review_rows = []
    decision_rows = []

    if offer_candidates_df is not None and not offer_candidates_df.empty:
        for _, c in offer_candidates_df.iterrows():
            needs_review = bool(c.get("needs_review"))
            if needs_review:
                review_id = make_hash_id("review", "candidate", c["candidate_id"])
                review_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "review_id": review_id,
                    "candidate_id": c["candidate_id"],
                    "grid_row_id": None,
                    "page_num": int(c["page_num"]),
                    "section_id": c["section_id"],
                    "review_reason": c.get("validation_errors_json") or "low_confidence_or_incomplete_candidate",
                    "review_status": "pending",
                    "source_table": "silver_offerCandidates",
                    "raw_payload_json": safe_json(c.to_dict()),
                    "created_ts": NOW_TS(),
                })

    if price_grid_rows_df is not None and not price_grid_rows_df.empty:
        for _, g in price_grid_rows_df.iterrows():
            needs_review = bool(g.get("needs_review"))
            if needs_review:
                review_id = make_hash_id("review", "grid", g["grid_row_id"])
                review_rows.append({
                    "deck_id": deck_id,
                    "run_id": run_id,
                    "pdf_hash": pdf_hash,
                    "pdf_name": pdf_name,
                    "deck_week": deck_week,
                    "review_id": review_id,
                    "candidate_id": None,
                    "grid_row_id": g["grid_row_id"],
                    "page_num": int(g["page_num"]),
                    "section_id": g["section_id"],
                    "review_reason": g.get("validation_errors_json") or "low_confidence_or_incomplete_price_grid_row",
                    "review_status": "pending",
                    "source_table": "silver_priceGridRows",
                    "raw_payload_json": safe_json(g.to_dict()),
                    "created_ts": NOW_TS(),
                })

    if normalized_offers_df is not None and not normalized_offers_df.empty:
        auto_df = normalized_offers_df[normalized_offers_df["auto_approve_flag"] == True].copy()
        for _, o in auto_df.iterrows():
            decision_id = make_hash_id("decision", "auto", o["candidate_id"])
            decision_rows.append({
                "deck_id": deck_id,
                "run_id": run_id,
                "pdf_hash": pdf_hash,
                "pdf_name": pdf_name,
                "deck_week": deck_week,
                "decision_id": decision_id,
                "candidate_id": o["candidate_id"],
                "grid_row_id": None,
                "page_num": int(o["page_num"]),
                "section_id": o["section_id"],
                "decision_type": "auto_approved",
                "decision_status": "approved",
                "decision_source": "system_rules_v2",
                "decision_notes": f"confidence={o.get('confidence')}; auto_approve_threshold={AUTO_APPROVE_MIN_CONFIDENCE}",
                "created_ts": NOW_TS(),
            })

    review_df = pd.DataFrame(review_rows)
    decisions_df = pd.DataFrame(decision_rows)

    print(f"review queue rows: {len(review_df):,}")
    print(f"auto-approved decision rows: {len(decisions_df):,}")

    return review_df, decisions_df


# ======================================================================================
# SECTION 16 - PIPELINE RUNNER
# ======================================================================================

def main():
    print("\n" + "=" * 100)
    print("STARTING COMPETITIVE OFFER REPORT PIPELINE V2")
    print("=" * 100)

    pdf_path = choose_pdf_path()
    pdf_name = os.path.basename(pdf_path)
    pdf_hash = compute_file_hash(pdf_path)
    file_size_bytes = os.path.getsize(pdf_path)
    run_id = f"run_{uuid.uuid4().hex[:16]}"

    # Open first page once for deck week.
    temp_doc = fitz.open(pdf_path)
    first_page_text = temp_doc[0].get_text("text") if temp_doc.page_count else ""
    page_count = temp_doc.page_count
    temp_doc.close()

    deck_week = infer_deck_week_from_pdf_or_filename(pdf_path, first_page_text)
    deck_id = f"deck_{deck_week.strftime('%Y%m%d')}_{pdf_hash[:12]}"

    print(f"pdf_name: {pdf_name}")
    print(f"pdf_hash: {pdf_hash}")
    print(f"deck_week: {deck_week}")
    print(f"deck_id: {deck_id}")
    print(f"run_id: {run_id}")
    print(f"page_count: {page_count}")

    duplicate_action = "new"
    existing_df = get_existing_decks_by_hash(pdf_hash)
    if not existing_df.empty:
        duplicate_action = resolve_duplicate_action(existing_df)
        if duplicate_action == "replace":
            existing_deck_ids = existing_df["deck_id"].dropna().unique().tolist()
            delete_existing_pdf_rows(pdf_hash, existing_deck_ids)
    else:
        duplicate_action = "new"

    # Bronze deck row.
    pdf_decks_df = pd.DataFrame([{
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "pdf_name": pdf_name,
        "deck_week": deck_week,
        "pdf_path": pdf_path,
        "page_count": page_count,
        "file_size_bytes": file_size_bytes,
        "duplicate_action": duplicate_action,
        "pipeline_version": "v2_title_first_vision_routing",
        "ingestion_ts": NOW_TS(),
    }])

    # Extract bronze.
    slide_raw_df, slide_lines_df = extract_pdf_bronze(
        pdf_path=pdf_path,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        deck_week=deck_week,
    )

    detected_entities_df = detect_entities(
        slide_raw_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # TOC as QA.
    toc_map_df = parse_toc_page_map(
        slide_raw_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Classify slides.
    slide_sections_df = classify_all_slides(
        slide_raw_df=slide_raw_df,
        slide_lines_df=slide_lines_df,
        toc_map_df=toc_map_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Content blocks.
    content_blocks_df = build_content_blocks(
        slide_raw_df=slide_raw_df,
        slide_sections_df=slide_sections_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Deterministic offer candidates.
    deterministic_candidates_df = build_offer_candidates_from_blocks(
        content_blocks_df=content_blocks_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Gemini Vision extraction.
    slide_tables_df, vision_candidates_df, price_grid_rows_df = run_vision_extraction(
        slide_raw_df=slide_raw_df,
        slide_sections_df=slide_sections_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Combine candidates.
    candidate_frames = []
    if deterministic_candidates_df is not None and not deterministic_candidates_df.empty:
        candidate_frames.append(deterministic_candidates_df)
    if vision_candidates_df is not None and not vision_candidates_df.empty:
        candidate_frames.append(vision_candidates_df)

    if candidate_frames:
        offer_candidates_df = pd.concat(candidate_frames, ignore_index=True)
        offer_candidates_df = offer_candidates_df.drop_duplicates(subset=["candidate_id"])
    else:
        offer_candidates_df = pd.DataFrame()

    print(f"combined offer_candidates rows: {len(offer_candidates_df):,}")

    # Normalize.
    normalized_offers_df = normalize_offers(
        offer_candidates_df=offer_candidates_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Device bridge.
    device_bridge_df = build_device_bridge(
        normalized_offers_df=normalized_offers_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Review and decisions.
    review_df, decisions_df = build_review_tables(
        offer_candidates_df=offer_candidates_df,
        price_grid_rows_df=price_grid_rows_df,
        normalized_offers_df=normalized_offers_df,
        deck_id=deck_id,
        run_id=run_id,
        pdf_hash=pdf_hash,
        pdf_name=pdf_name,
        deck_week=deck_week,
    )

    # Write to BigQuery.
    if WRITE_TO_BQ:
        print("\n" + "=" * 100)
        print("LOADING TO BIGQUERY")
        print("=" * 100)

        append_df_to_bq(pdf_decks_df, "bronze_pdfDecks")
        append_df_to_bq(slide_raw_df, "bronze_slideRaw")
        append_df_to_bq(slide_lines_df, "bronze_slideLines")
        append_df_to_bq(detected_entities_df, "bronze_detectedEntities")
        append_df_to_bq(slide_tables_df, "bronze_slideTables")

        append_df_to_bq(toc_map_df, "silver_tocPageMap")
        append_df_to_bq(slide_sections_df, "silver_slideSections")
        append_df_to_bq(content_blocks_df, "silver_slideContentBlocks")
        append_df_to_bq(offer_candidates_df, "silver_offerCandidates")
        append_df_to_bq(normalized_offers_df, "silver_normalizedOffers")
        append_df_to_bq(device_bridge_df, "silver_offerDeviceBridge")
        append_df_to_bq(price_grid_rows_df, "silver_priceGridRows")

        append_df_to_bq(review_df, "review_extractionReview")
        append_df_to_bq(decisions_df, "review_reviewDecisions")

    print("\n" + "=" * 100)
    print("FINAL RUN SUMMARY")
    print("=" * 100)
    summary = {
        "pdf_name": pdf_name,
        "deck_week": str(deck_week),
        "deck_id": deck_id,
        "run_id": run_id,
        "pdf_hash": pdf_hash,
        "duplicate_action": duplicate_action,
        "page_count": page_count,
        "bronze_slide_raw_rows": len(slide_raw_df),
        "bronze_slide_line_rows": len(slide_lines_df),
        "detected_entity_rows": len(detected_entities_df),
        "toc_map_rows": len(toc_map_df),
        "slide_section_rows": len(slide_sections_df),
        "content_block_rows": len(content_blocks_df),
        "vision_table_rows": len(slide_tables_df),
        "offer_candidate_rows": len(offer_candidates_df),
        "normalized_offer_rows": len(normalized_offers_df),
        "device_bridge_rows": len(device_bridge_df),
        "price_grid_rows": len(price_grid_rows_df),
        "review_queue_rows": len(review_df),
        "auto_approved_decision_rows": len(decisions_df),
    }

    for k, v in summary.items():
        print(f"{k:<35}: {v}")

    if PRINT_SAMPLE_ROWS:
        print("\n" + "=" * 100)
        print("SAMPLE OUTPUTS")
        print("=" * 100)

        print("\nSlide sections:")
        display(slide_sections_df[[
            "page_num", "section_id", "layout_type", "classifier_method",
            "toc_expected_section", "toc_mismatch_flag", "title_header_text"
        ]].head(40))

        if not offer_candidates_df.empty:
            print("\nOffer candidates sample:")
            display(offer_candidates_df[[
                "page_num", "section_id", "carrier", "product_name", "offer_type",
                "offer_amount_text", "confidence", "needs_review", "raw_evidence"
            ]].head(30))

        if not price_grid_rows_df.empty:
            print("\nPrice grid rows sample:")
            display(price_grid_rows_df.head(30))

        if not review_df.empty:
            print("\nReview queue sample:")
            display(review_df[[
                "page_num", "section_id", "review_reason", "source_table"
            ]].head(30))

    print("\nPipeline complete.")

    return {
        "summary": summary,
        "pdf_decks_df": pdf_decks_df,
        "slide_raw_df": slide_raw_df,
        "slide_lines_df": slide_lines_df,
        "detected_entities_df": detected_entities_df,
        "toc_map_df": toc_map_df,
        "slide_sections_df": slide_sections_df,
        "content_blocks_df": content_blocks_df,
        "slide_tables_df": slide_tables_df,
        "offer_candidates_df": offer_candidates_df,
        "normalized_offers_df": normalized_offers_df,
        "device_bridge_df": device_bridge_df,
        "price_grid_rows_df": price_grid_rows_df,
        "review_df": review_df,
        "decisions_df": decisions_df,
    }


# ======================================================================================
# SECTION 17 - RUN
# ======================================================================================

pipeline_outputs = main()